# Cognifyz Technologies - Data Analysis Internship
## Level 1

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("Dataset_.csv", encoding="utf-8-sig")
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

---
## Task 1 - Top Cuisines
**Objective:** Determine the top 3 most common cuisines and calculate the percentage of restaurants serving each.

In [ ]:
total = len(df)
cuisine_series = df["Cuisines"].dropna().str.split(", ")
all_cuisines = cuisine_series.explode()
cuisine_counts = all_cuisines.value_counts()
top3 = cuisine_counts.head(3)

print(f"Total Restaurants: {total}")
print("
Top 3 Most Common Cuisines:")
for rank, (cuisine, count) in enumerate(top3.items(), 1):
    pct = (count / total) * 100
    print(f"{rank}. {cuisine}: {count} restaurants ({pct:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top3_pcts = [(c / total) * 100 for c in top3.values]
colors = ["#4C72B0", "#DD8452", "#55A868"]
axes[0].pie(top3_pcts, labels=top3.index, autopct="%1.2f%%", colors=colors, startangle=140, textprops={"fontsize": 12})
axes[0].set_title("Top 3 Cuisines (% of Total Restaurants)", fontsize=13, fontweight="bold")

top10 = cuisine_counts.head(10)
top10_pcts = [(c / total) * 100 for c in top10.values]
bars = axes[1].bar(top10.index, top10_pcts, color=sns.color_palette("Blues_d", 10))
axes[1].set_title("Top 10 Cuisines by Percentage", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Cuisine")
axes[1].set_ylabel("Percentage (%)")
axes[1].set_xticklabels(top10.index, rotation=35, ha="right", fontsize=9)
for bar, val in zip(bars, top10_pcts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f"{val:.1f}%", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

---
## Task 2 - City Analysis
**Objective:** Identify the city with the highest number of restaurants, calculate average rating per city, and determine the city with the highest average rating.

In [ ]:
city_counts = df["City"].value_counts()
city_avg_rating = df.groupby("City")["Aggregate rating"].mean().round(2)
top_city = city_counts.idxmax()
city_rating_filtered = df.groupby("City").filter(lambda x: len(x) >= 10)
top_rated_city = city_rating_filtered.groupby("City")["Aggregate rating"].mean().idxmax()
top_rated_value = city_rating_filtered.groupby("City")["Aggregate rating"].mean().max()

print(f"City with most restaurants: {top_city} ({city_counts.max()})")
print(f"City with highest avg rating (min 10 restaurants): {top_rated_city} ({top_rated_value:.2f})")
print("
Top 10 Cities by Restaurant Count:")
print(city_counts.head(10).to_string())

In [ ]:
top15 = city_counts.head(15)
top15_avg = city_avg_rating[top15.index].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bars = axes[0].bar(top15.index, top15.values, color=sns.color_palette("Blues_d", 15))
axes[0].set_title("Top 15 Cities by Number of Restaurants", fontsize=13, fontweight="bold")
axes[0].set_xlabel("City")
axes[0].set_ylabel("Number of Restaurants")
axes[0].set_xticklabels(top15.index, rotation=45, ha="right", fontsize=8)
for bar, val in zip(bars, top15.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha="center", fontsize=7)

bars2 = axes[1].bar(top15_avg.index, top15_avg.values, color=sns.color_palette("Greens_d", 15))
axes[1].set_title("Avg Rating per City (Top 15)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("City")
axes[1].set_ylabel("Average Rating")
axes[1].set_ylim(0, 5)
axes[1].set_xticklabels(top15_avg.index, rotation=45, ha="right", fontsize=8)
for bar, val in zip(bars2, top15_avg.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f"{val:.2f}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()

---
## Task 3 - Price Range Distribution
**Objective:** Visualize the distribution of price ranges and calculate the percentage in each category.

In [ ]:
price_counts = df["Price range"].value_counts().sort_index()
price_pcts = (price_counts / total * 100).round(2)
labels = {1: "Range 1 (Low)", 2: "Range 2 (Medium)", 3: "Range 3 (High)", 4: "Range 4 (Very High)"}

print("Price Range Distribution:")
for pr in price_counts.index:
    print(f"  {labels[pr]}: {price_counts[pr]} restaurants ({price_pcts[pr]:.2f}%)")

In [ ]:
colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
bars = axes[0].bar([labels[i] for i in price_counts.index], price_counts.values, color=colors)
axes[0].set_title("Price Range Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Price Range")
axes[0].set_ylabel("Number of Restaurants")
for bar, pct in zip(bars, price_pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30, f"{pct:.2f}%", ha="center", fontsize=10, fontweight="bold")

axes[1].pie(price_pcts.values, labels=[labels[i] for i in price_counts.index], autopct="%1.2f%%", colors=colors, startangle=140, textprops={"fontsize": 11})
axes[1].set_title("Price Range % Share", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Task 4 - Online Delivery
**Objective:** Determine the percentage of restaurants offering online delivery and compare average ratings.

In [ ]:
delivery_counts = df["Has Online delivery"].value_counts()
delivery_pcts = (delivery_counts / total * 100).round(2)
avg_with = df[df["Has Online delivery"] == "Yes"]["Aggregate rating"].mean()
avg_without = df[df["Has Online delivery"] == "No"]["Aggregate rating"].mean()

print("Online Delivery Availability:")
for val in delivery_counts.index:
    print(f"  {val}: {delivery_counts[val]} restaurants ({delivery_pcts[val]:.2f}%)")
print(f"
Avg Rating WITH delivery:    {avg_with:.2f}")
print(f"Avg Rating WITHOUT delivery: {avg_without:.2f}")

In [ ]:
colors = ["#55A868", "#C44E52"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].pie(delivery_pcts.values, labels=[f"{v} Delivery" for v in delivery_counts.index], autopct="%1.2f%%", colors=colors, startangle=140, textprops={"fontsize": 11})
axes[0].set_title("Online Delivery Availability", fontsize=12, fontweight="bold")

bars = axes[1].bar(["With Delivery", "Without Delivery"], [avg_with, avg_without], color=colors, width=0.4)
axes[1].set_title("Avg Rating Comparison", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Average Rating")
axes[1].set_ylim(0, 5)
for bar, val in zip(bars, [avg_with, avg_without]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f"{val:.2f}", ha="center", fontsize=12, fontweight="bold")

data_yes = df[df["Has Online delivery"] == "Yes"]["Aggregate rating"]
data_no  = df[df["Has Online delivery"] == "No"]["Aggregate rating"]
bp = axes[2].boxplot([data_yes, data_no], labels=["With Delivery", "Without Delivery"], patch_artist=True)
bp["boxes"][0].set_facecolor("#55A868")
bp["boxes"][1].set_facecolor("#C44E52")
axes[2].set_title("Rating Distribution Boxplot", fontsize=12, fontweight="bold")
axes[2].set_ylabel("Aggregate Rating")
plt.tight_layout()
plt.show()